In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kimanaru/dataset-tien-xu-li/val_ml_clean_text.csv
/kaggle/input/datasets/kimanaru/dataset-tien-xu-li/train_ml_clean_text.csv
/kaggle/input/datasets/kimanaru/dataset-tien-xu-li/test_ml_clean_text.csv


In [2]:
import pandas as pd
from pathlib import Path

RESULT_DIR = Path("/kaggle/input/datasets/kimanaru/dataset-tien-xu-li")

train_df = pd.read_csv(RESULT_DIR / "train_ml_clean_text.csv")
val_df = pd.read_csv(RESULT_DIR / "val_ml_clean_text.csv")
test_df = pd.read_csv(RESULT_DIR / "test_ml_clean_text.csv")

train_df["ml_clean_text"] = train_df["ml_clean_text"].fillna("").astype(str)
val_df["ml_clean_text"] = val_df["ml_clean_text"].fillna("").astype(str)
test_df["ml_clean_text"] = test_df["ml_clean_text"].fillna("").astype(str)

train_df["category"] = train_df["category"].astype(str)
val_df["category"] = val_df["category"].astype(str)
test_df["category"] = test_df["category"].astype(str)

X_train = train_df["ml_clean_text"]
y_train = train_df["category"]

X_val = val_df["ml_clean_text"]
y_val = val_df["category"]

X_test = test_df["ml_clean_text"]
y_test = test_df["category"]

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time

# =========================
# Cấu hình TF-IDF cuối cùng
# =========================
tfidf_vectorizer = TfidfVectorizer(
    max_features= 300000,
    ngram_range=(1, 2),
    max_df=0.8,
    sublinear_tf=True
)

In [4]:
# =========================
# Trích xuất đặc trưng TF-IDF
# =========================

# Fit chỉ trên train để tránh data leakage
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Validation và test chỉ transform
X_val_tfidf = tfidf_vectorizer.transform(X_val)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("X_train_tfidf:", X_train_tfidf.shape)
print("X_val_tfidf:", X_val_tfidf.shape)
print("X_test_tfidf:", X_test_tfidf.shape)

print("Number of TF-IDF features:", len(tfidf_vectorizer.get_feature_names_out()))


X_train_tfidf: (103685, 300000)
X_val_tfidf: (22218, 300000)
X_test_tfidf: (22219, 300000)
Number of TF-IDF features: 300000


In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)
y_test_enc = le.transform(y_test)

In [6]:
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import f1_score
import pandas as pd
import time

# Tạo Pool một lần, tránh xử lý lại dữ liệu nhiều lần
train_pool = Pool(X_train_tfidf, y_train)
val_pool = Pool(X_val_tfidf, y_val)

param_list = [
    {"depth": 4, "learning_rate": 0.1, "l2_leaf_reg": 5},
    {"depth": 5, "learning_rate": 0.1, "l2_leaf_reg": 5},
    {"depth": 6, "learning_rate": 0.15, "l2_leaf_reg": 10},
]

results = []
best_model = None
best_f1 = -1

for i, params in enumerate(param_list):
    print(f"\n===== Thử nghiệm {i+1}/{len(param_list)} =====")
    print(params)

    start_time = time.time()

    model = CatBoostClassifier(
        iterations=50,                 # giảm từ 100
        depth=params["depth"],          # depth thấp hơn chạy nhanh hơn nhiều
        learning_rate=params["learning_rate"],
        l2_leaf_reg=params["l2_leaf_reg"],

        loss_function="MultiClass",
        eval_metric="TotalF1",

        auto_class_weights="Balanced",
        early_stopping_rounds=10,       # dừng sớm hơn

        random_seed=42,
        verbose=10,

        thread_count=-1,                # dùng toàn bộ CPU
        bootstrap_type="Bernoulli",
        subsample=0.8,                  # train nhanh hơn
        rsm=0.8,                        # dùng 80% feature mỗi split
        max_ctr_complexity=1,
        allow_writing_files=False
    )

    model.fit(
        train_pool,
        eval_set=val_pool,
        use_best_model=True
    )

    y_val_pred = model.predict(val_pool)

    val_macro_f1 = f1_score(
        y_val,
        y_val_pred,
        average="macro"
    )

    elapsed_time = time.time() - start_time

    results.append({
        **params,
        "val_macro_f1": val_macro_f1,
        "best_iteration": model.get_best_iteration(),
        "time_seconds": elapsed_time
    })

    if val_macro_f1 > best_f1:
        best_f1 = val_macro_f1
        best_model = model

results_df = pd.DataFrame(results).sort_values(
    "val_macro_f1",
    ascending=False
)

print(results_df)
print("Best validation macro F1:", best_f1)


===== Thử nghiệm 1/3 =====
{'depth': 4, 'learning_rate': 0.1, 'l2_leaf_reg': 5}
0:	learn: 0.1178544	test: 0.1183404	best: 0.1183404 (0)	total: 8.1s	remaining: 6m 37s
10:	learn: 0.2619757	test: 0.2700328	best: 0.2766700 (9)	total: 1m 3s	remaining: 3m 44s
20:	learn: 0.3522333	test: 0.3614307	best: 0.3614307 (20)	total: 1m 57s	remaining: 2m 42s
30:	learn: 0.3853479	test: 0.3909289	best: 0.3909289 (30)	total: 2m 52s	remaining: 1m 45s
40:	learn: 0.3967426	test: 0.4013194	best: 0.4013194 (40)	total: 3m 46s	remaining: 49.8s
49:	learn: 0.4163932	test: 0.4187607	best: 0.4187607 (49)	total: 4m 35s	remaining: 0us

bestTest = 0.4187606928
bestIteration = 49


===== Thử nghiệm 2/3 =====
{'depth': 5, 'learning_rate': 0.1, 'l2_leaf_reg': 5}
0:	learn: 0.1219262	test: 0.1230013	best: 0.1230013 (0)	total: 12.3s	remaining: 10m 2s
10:	learn: 0.2697139	test: 0.2776273	best: 0.2776273 (10)	total: 1m 37s	remaining: 5m 45s
20:	learn: 0.3419631	test: 0.3486976	best: 0.3486976 (20)	total: 3m 2s	remaining: 4m 1

In [7]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

# Dự đoán trên tập test bằng model tốt nhất
y_test_pred = best_model.predict(X_test_tfidf)

test_accuracy = accuracy_score(y_test, y_test_pred)
test_macro_precision = precision_score(y_test, y_test_pred, average="macro", zero_division=0)
test_macro_recall = recall_score(y_test, y_test_pred, average="macro", zero_division=0)
test_macro_f1 = f1_score(y_test, y_test_pred, average="macro")

print("\n===== KẾT QUẢ TRÊN TẬP TEST =====")
print("Test Accuracy:", test_accuracy)
print("Test Macro Precision:", test_macro_precision)
print("Test Macro Recall:", test_macro_recall)
print("Test Macro F1:", test_macro_f1)

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, zero_division=0))


===== KẾT QUẢ TRÊN TẬP TEST =====
Test Accuracy: 0.4683829155227508
Test Macro Precision: 0.501805505445268
Test Macro Recall: 0.4505248132435939
Test Macro F1: 0.45708144163287084

Classification Report:
                precision    recall  f1-score   support

  BLACK VOICES       0.42      0.38      0.40       687
      BUSINESS       0.46      0.37      0.41       899
        COMEDY       0.44      0.36      0.39       810
 ENTERTAINMENT       0.24      0.65      0.35      2605
  FOOD & DRINK       0.59      0.50      0.54       951
HEALTHY LIVING       0.19      0.18      0.18      1004
 HOME & LIVING       0.45      0.54      0.49       648
     PARENTING       0.50      0.33      0.40      1319
       PARENTS       0.26      0.43      0.33       593
      POLITICS       0.89      0.51      0.65      5341
  QUEER VOICES       0.82      0.61      0.70       952
        SPORTS       0.46      0.48      0.47       761
STYLE & BEAUTY       0.60      0.69      0.64      1472
        T